## 3.0 Step 1: Exploration of Annotated Samples

### Objective

The first step in the pseudo-labeling pipeline is to thoroughly explore the **annotated samples**. The goal of this exploration is to:

- Understand the distribution and frequency of the different tissue classes present in the annotations.
- Identify which fine-grained labels exist and how they relate to our broader objective of **Tumor vs Non-Tumor** classification.
- Determine the most discriminative marker genes associated with each tissue class.
- Decide on a practical label taxonomy that we can use to generate reliable pseudo-labels for the remaining unannotated samples.

This exploration will guide the design of our pseudo-labeling strategy in the subsequent steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ============================================================
# 4. Pseudo-Label Generation
# ============================================================

import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from tqdm import tqdm

# ====================== USER CONFIGURATION ======================
# Update this path to point to your annotation folder
ANNOTATION_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/raw/Annotation")

# Optional: Path to your processed AnnData files (we may use later)
PROCESSED_ST_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train")
PROCESSED_ST_DIR1 = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/val")
PROCESSED_ST_DIR2 = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/test")
# ================================================================

print("Annotation directory exists:", ANNOTATION_DIR.exists())
print("Number of annotation files found:", len(list(ANNOTATION_DIR.glob("*.csv"))))

Annotation directory exists: True
Number of annotation files found: 24


### Load and Explore all Annotation Files



In [ ]:
# ============================================================
# Load all annotation files and extract labels
# ============================================================

annotation_files = sorted(list(ANNOTATION_DIR.glob("*.csv")))

print(f"Found {len(annotation_files)} annotation files.\n")

all_annotations = []
sample_label_summary = []

for file_path in tqdm(annotation_files, desc="Loading annotation files"):
    try:
        df = pd.read_csv(file_path)

        # Try to detect the label column
        label_col = None
        possible_label_cols = ['label', 'label_new', 'Annotations', 'annot_type', 'fine_annot_type', 'cell_type', 'category']

        for col in possible_label_cols:
            if col in df.columns:
                label_col = col
                break

        if label_col is None:
            print(f"Warning: No label column found in {file_path.name}. Skipping.")
            continue

        # Get sample name from filename (remove extension)
        sample_name = file_path.stem

        # Store per-sample summary
        label_counts = df[label_col].value_counts().to_dict()
        sample_label_summary.append({
            "sample_name": sample_name,
            "total_spots": len(df),
            "unique_labels": df[label_col].nunique(),
            "label_counts": label_counts
        })

        # Store all annotations with sample info
        temp_df = df[[label_col]].copy()
        temp_df['sample_name'] = sample_name
        temp_df = temp_df.rename(columns={label_col: 'original_label'})
        all_annotations.append(temp_df)

    except Exception as e:
        print(f"Error reading {file_path.name}: {e}")

# Combine all annotations
if all_annotations:
    annotations_df = pd.concat(all_annotations, ignore_index=True)
    print(f"\nTotal annotated spots across all files: {len(annotations_df)}")
else:
    annotations_df = pd.DataFrame()
    print("No annotations loaded.")

Found 24 annotation files.



Loading annotation files: 100%|██████████| 24/24 [00:00<00:00, 118.79it/s]


Total annotated spots across all files: 34849


### Summary of Unique Tissue Classes

In [ ]:
# ============================================================
# Summary of unique tissue classes
# ============================================================

if not annotations_df.empty:
    print("=" * 70)
    print("UNIQUE TISSUE CLASSES FOUND IN ANNOTATED SAMPLES")
    print("=" * 70)

    # Convert all unique labels to strings to handle mixed types (e.g., nan and strings)
    unique_labels = annotations_df['original_label'].astype(str).unique()
    print(f"\nTotal unique labels: {len(unique_labels)}\n")

    for label in sorted(unique_labels):
        # Ensure comparison is with string representation if original_label might contain non-strings
        count = (annotations_df['original_label'].astype(str) == label).sum()
        print(f"  - {label}: {count} spots")

    # Create a clean summary DataFrame
    label_summary = annotations_df['original_label'].value_counts(dropna=False).reset_index()
    label_summary.columns = ['Tissue Class', 'Total Spots']
    label_summary['Percentage'] = (label_summary['Total Spots'] / label_summary['Total Spots'].sum() * 100).round(2)

    print("\n" + "=" * 70)
    print("LABEL DISTRIBUTION (All Annotated Samples Combined)")
    print("=" * 70)
    print(label_summary.to_string(index=False))

UNIQUE TISSUE CLASSES FOUND IN ANNOTATED SAMPLES

Total unique labels: 40

  - Adipose tissue: 911 spots
  - Artefacts: 215 spots
  - Artifacts: 2238 spots
  - Endothelial: 16 spots
  - Fibrosis: 3123 spots
  - Fibrosis (peritumoral): 205 spots
  - Fibrous stroma: 135 spots
  - Healthy: 485 spots
  - High TILs Stroma: 529 spots
  - Hyperplasia: 11 spots
  - In Situ Carcinoma: 129 spots
  - In situ Carcinoma*: 220 spots
  - Invasive: 1663 spots
  - Lymphocytes: 527 spots
  - Lymphoid stroma: 378 spots
  - Necrosis: 98 spots
  - Non Tumor: 1843 spots
  - Normal Epithelium: 271 spots
  - Peripheral Nerve: 24 spots
  - Surrounding tumor: 823 spots
  - Tumor: 1502 spots
  - Tumor Cells: 1083 spots
  - Tumor Stroma: 776 spots
  - Tumor cells: 1431 spots
  - Tumor cells - Spindle Cells: 1413 spots
  - Tumor cells ?: 4 spots
  - Tumor stroma fibrous: 206 spots
  - Tumor stroma with inflammation: 1104 spots
  - Tumour Cells: 2736 spots
  - Tumour Stroma: 214 spots
  - Tumour cells: 2056 spots
 

### Visualization of class Distribution

In [ ]:
# ============================================================
# Visualization
# ============================================================

if not annotations_df.empty:
    plt.figure(figsize=(12, 6))

    sns.countplot(
        data=annotations_df,
        y='original_label',
        order=annotations_df['original_label'].value_counts().index,
        palette='viridis'
    )

    plt.title("Distribution of Tissue Classes in Annotated Samples", fontsize=14, fontweight='bold')
    plt.xlabel("Number of Spots")
    plt.ylabel("Tissue Class")
    plt.tight_layout()

    # Save figure
    save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
    save_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_dir / "annotated_class_distribution.png", dpi=1200, bbox_inches='tight')
    plt.show()

    print(f"\nFigure saved to: {save_dir / 'annotated_class_distribution.png'}")

### Per Sample Label Summary

In [ ]:
# ============================================================
# Per-sample label summary
# ============================================================

if sample_label_summary:
    sample_summary_df = pd.DataFrame(sample_label_summary)
    print("\nPer-sample annotation summary (first 5 samples):")
    print(sample_summary_df[['sample_name', 'total_spots', 'unique_labels']].head())

    # You can save this for later use
    sample_summary_df.to_csv(save_dir / "annotated_samples_summary.csv", index=False)
    print(f"\nPer-sample summary saved.")

### Create Label Mapping

In [ ]:
# ============================================================
# Create Label Mapping Dictionary
# ============================================================

label_mapping = {
    # === TUMOR ===
    'Invasive': 'Tumor',
    'invasive cancer': 'Tumor',
    'Tumor': 'Tumor',
    'Tumour Cells': 'Tumor',
    'Tumor cells': 'Tumor',
    'Tumor Cells': 'Tumor',
    'Tumour cells': 'Tumor',
    'Tumor cells - Spindle Cells': 'Tumor',
    'cancer in situ': 'Tumor',
    'In Situ Carcinoma': 'Tumor',
    'In situ Carcinoma*': 'Tumor',

    # === TUMOR STROMA / FIBROSIS ===
    'Fibrosis': 'Tumor Stroma',
    'Fibrosis (peritumoral)': 'Tumor Stroma',
    'Tumor Stroma': 'Tumor Stroma',
    'Tumour Stroma': 'Tumor Stroma',
    'Tumor stroma fibrous': 'Tumor Stroma',
    'Tumor stroma with inflammation': 'Tumor Stroma',
    'connective tissue': 'Tumor Stroma',
    'Fibrous stroma': 'Tumor Stroma',

    # === NON-TUMOR / NORMAL ===
    'Non Tumor': 'Non-Tumor',
    'Healthy': 'Non-Tumor',
    'Normal Epithelium': 'Non-Tumor',
    'Adipose tissue': 'Non-Tumor',
    'adipose tissue': 'Non-Tumor',
    'breast glands': 'Non-Tumor',

    # === IMMUNE / LYMPHOID ===
    'Lymphocytes': 'Immune',
    'Lymphoid stroma': 'Immune',
    'High TILs Stroma': 'Immune',
    'immune infiltrate': 'Immune',

    # === SURROUNDING / OTHER ===
    'Surrounding tumor': 'Surrounding Tumor',
    'undetermined': 'Other',

    # === EXCLUDE (will be marked separately) ===
    'nan': 'Exclude',
    'Artifacts': 'Exclude',
    'Artefacts': 'Exclude',
    'Necrosis': 'Exclude',
    'Vascular': 'Exclude',
    'Peripheral Nerve': 'Exclude',
    'Endothelial': 'Exclude',
    'Hyperplasia': 'Exclude',
    'Tumor cells ?': 'Exclude',
}

print("Label mapping dictionary created with", len(label_mapping), "entries.")

Label mapping dictionary created with 40 entries.


### Apply the Mapping

In [ ]:
# ============================================================
# Apply Mapping to Create Clean Tissue Groups
# ============================================================

annotations_df['tissue_group'] = annotations_df['original_label'].map(label_mapping)

# Check unmapped labels
unmapped = annotations_df[annotations_df['tissue_group'].isna()]['original_label'].unique()
if len(unmapped) > 0:
    print("\n⚠️  Unmapped labels found:")
    for label in unmapped:
        print(f"   - {label}")
else:
    print("\n✅ All labels successfully mapped.")

# Summary of new groups
print("\n" + "="*60)
print("DISTRIBUTION AFTER GROUPING")
print("="*60)
group_summary = annotations_df['tissue_group'].value_counts()
print(group_summary)


⚠️  Unmapped labels found:
   - nan

DISTRIBUTION AFTER GROUPING
tissue_group
Tumor                14133
Tumor Stroma          6382
Non-Tumor             3910
Exclude               2634
Immune                1687
Surrounding Tumor      823
Other                  324
Name: count, dtype: int64


### Visualization of Grouped Tissue Classes

In [ ]:
# ============================================================
# Visualization of Grouped Tissue Classes
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Create save directory
save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
save_dir.mkdir(parents=True, exist_ok=True)

# Define a clean color palette for the groups
group_colors = {
    'Tumor': '#E63946',           # Red
    'Tumor Stroma': '#F4A261',    # Orange
    'Non-Tumor': '#2A9D8F',       # Teal/Green
    'Immune': '#457B9D',          # Blue
    'Surrounding Tumor': '#9B5DE5', # Purple
    'Other': '#6C757D',           # Gray
    'Exclude': '#ADB5BD'          # Light gray
}

# Get counts
group_counts = annotations_df['tissue_group'].value_counts().reset_index()
group_counts.columns = ['Tissue Group', 'Number of Spots']

# Calculate percentages
total_spots = group_counts['Number of Spots'].sum()
group_counts['Percentage'] = (group_counts['Number of Spots'] / total_spots * 100).round(2)

print("=" * 70)
print("GROUPED TISSUE CLASS DISTRIBUTION")
print("=" * 70)
print(group_counts.to_string(index=False))

# Create horizontal bar plot
plt.figure(figsize=(11, 7))

# Reorder for better visualization (Tumor-related on top)
order = ['Tumor', 'Tumor Stroma', 'Non-Tumor', 'Immune', 'Surrounding Tumor', 'Other', 'Exclude']

sns.barplot(
    data=group_counts,
    y='Tissue Group',
    x='Number of Spots',
    order=order,
    palette=[group_colors.get(g, '#6C757D') for g in order]
)

plt.title("Distribution of Grouped Tissue Classes in Annotated Samples\n(After Mapping)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Number of Spots", fontsize=12)
plt.ylabel("Tissue Group", fontsize=12)

# Add value labels on bars
for i, (value, pct) in enumerate(zip(group_counts.set_index('Tissue Group').loc[order]['Number of Spots'],
                                       group_counts.set_index('Tissue Group').loc[order]['Percentage'])):
    plt.text(value + 50, i, f"{value:,} ({pct:.1f}%)", va='center', fontsize=10)

plt.tight_layout()

# Save figure
save_path = save_dir / "grouped_tissue_classes_distribution.png"
plt.savefig(save_path, dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Figure saved to: {save_path}")

### Merge Annotations into ST files

In [ ]:
pip install scanpy

In [ ]:
# ============================================================
# Merge Annotations - Handles Variable Column Names
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np

# ====================== PATHS ======================
RAW_DATA_ROOT = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/raw/train")
PROCESSED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train")
OUTPUT_DIR    = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_labeled")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Possible column names (in order of priority)
POSSIBLE_ID_COLS   = ['name', 'V1', 'ID', 'barcode', 'spot_id', 'index']
POSSIBLE_LABEL_COLS = ['label', 'label_new', 'Annotations', 'annot_type', 'fine_annot_type', 'cell_type', 'category']

sample_folders = [f for f in RAW_DATA_ROOT.iterdir() if f.is_dir()]
print(f"Found {len(sample_folders)} sample folders.\n")

successfully_merged = []
failed = []
merge_method_used = {}

for sample_folder in tqdm(sample_folders, desc="Merging annotations"):
    sample_name = sample_folder.name
    annotation_path = sample_folder / f"{sample_name}_anno.csv"

    if not annotation_path.exists():
        continue

    adata_path = PROCESSED_DIR / f"{sample_name}.h5ad"
    if not adata_path.exists():
        failed.append(sample_name)
        continue

    try:
        # Load annotation
        annot_df = pd.read_csv(annotation_path)

        # === Find Spot ID column ===
        id_col = None
        for col in POSSIBLE_ID_COLS:
            if col in annot_df.columns:
                id_col = col
                break

        # === Find Label column ===
        label_col = None
        for col in POSSIBLE_LABEL_COLS:
            if col in annot_df.columns:
                label_col = col
                break

        if label_col is None:
            print(f"  Warning: No label column found in {sample_name}_anno.csv")
            failed.append(sample_name)
            continue

        # Load AnnData
        adata = sc.read_h5ad(adata_path)
        n_adata = adata.n_obs
        n_anno  = len(annot_df)

        # === Method 1: ID-based alignment (preferred) ===
        if id_col is not None:
            annot_df = annot_df.set_index(id_col)
            common_spots = adata.obs_names.intersection(annot_df.index)

            if len(common_spots) > 0:
                annot_aligned = annot_df.loc[common_spots]
                adata.obs['original_label'] = annot_aligned[label_col].values
                adata.obs['tissue_group']   = annot_aligned[label_col].map(label_mapping).values

                output_path = OUTPUT_DIR / f"{sample_name}.h5ad"
                adata.write_h5ad(output_path)

                successfully_merged.append(sample_name)
                merge_method_used[sample_name] = "ID-based"
                continue

        # === Method 2: Positional fallback (if ID matching failed but difference is small) ===
        diff = abs(n_adata - n_anno)
        if diff <= 20:   # Small mismatch threshold
            # Take only the first n_adata rows from annotation (exclude extra spots)
            annot_aligned = annot_df.iloc[:n_adata].copy()

            adata.obs['original_label'] = annot_aligned[label_col].values
            adata.obs['tissue_group']   = annot_aligned[label_col].map(label_mapping).values

            output_path = OUTPUT_DIR / f"{sample_name}.h5ad"
            adata.write_h5ad(output_path)

            successfully_merged.append(sample_name)
            merge_method_used[sample_name] = f"Positional (diff={diff})"
        else:
            print(f"  Skipped {sample_name}: Large mismatch ({n_adata} vs {n_anno}) and no ID column match.")
            failed.append(sample_name)

    except Exception as e:
        print(f"  Error on {sample_name}: {e}")
        failed.append(sample_name)

# ====================== SUMMARY ======================
print("\n" + "="*75)
print("ANNOTATION MERGING SUMMARY")
print("="*75)
print(f"Successfully merged : {len(successfully_merged)} samples")
print(f"Failed / Skipped    : {len(failed)} samples")

if successfully_merged:
    print("\nSuccessfully merged samples:")
    for s in successfully_merged:
        method = merge_method_used.get(s, "Unknown")
        print(f"  - {s}  [{method}]")

## 3.0 Step 2: Marker Gene Identification

### Objective

To:

1. Combine all successfully labeled annotated samples into a single AnnData object.
2. Perform differential expression analysis to identify genes that are significantly upregulated in each tissue group compared to others.
3. Extract and examine the top marker genes for each group.
4. Use these genes as the basis for defining pseudo-labeling rules in the subsequent steps.

This stage bridges the high-quality annotated data with the much larger set of unannotated samples, enabling us to scale our labels across the full dataset.


### Load all Labeled Annotated Samples

In [ ]:
import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Path to the labeled AnnData files
LABELED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_labeled")

# Get all labeled files
labeled_files = sorted(list(LABELED_DIR.glob("*.h5ad")))
print(f"Found {len(labeled_files)} labeled annotated samples.")

# Load and combine them
adata_list = []

for file_path in tqdm(labeled_files, desc="Loading labeled samples"):
    adata = sc.read_h5ad(file_path)

    # Keep only spots with valid tissue_group (exclude "Exclude" and NaN)
    adata = adata[adata.obs['tissue_group'].notna()].copy()
    adata = adata[adata.obs['tissue_group'] != 'Exclude'].copy()

    if adata.n_obs > 0:
        adata_list.append(adata)

# Concatenate all samples
adata_annotated = sc.concat(adata_list, join='outer', label='sample_name')
print(f"\nTotal spots for marker gene analysis: {adata_annotated.n_obs}")
print(f"Tissue groups present: {adata_annotated.obs['tissue_group'].unique().tolist()}")

Found 0 labeled annotated samples.


Loading labeled samples: 0it [00:00, ?it/s]


ValueError: No objects to concatenate

### Identify Marker Genes

In [ ]:
# ============================================================
# Marker Gene Identification
# ============================================================

# Run differential expression test
sc.tl.rank_genes_groups(
    adata_annotated,
    groupby='tissue_group',
    method='wilcoxon',        # Good balance of speed and accuracy
    n_genes=100,              # Get top 100 genes per group
    use_raw=False
)

# View top marker genes for each group
print("\nTop marker genes per tissue group:")
sc.pl.rank_genes_groups(
    adata_annotated,
    n_genes=8,
    sharey=False,
    figsize=(14, 10)
)

### Get Detailed Marker Gene Table

In [ ]:
# Extract top marker genes into a clean DataFrame
marker_genes = sc.get.rank_genes_groups_df(adata_annotated, group=None)

print("\nTop 15 marker genes for each tissue group:")
print(marker_genes.groupby('group').head(15).to_string())

In [ ]:
# ============================================================
# Update tissue_group with Refined Grouping
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm

LABELED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_labeled")

# New refined mapping
refined_mapping = {
    'Tumor': 'Tumor',
    'Surrounding Tumor': 'Tumor',
    'Tumor Stroma': 'Tumor Stroma',
    'Non-Tumor': 'Non-Tumor',
    'Immune': 'Non-Tumor',
    'Other': 'Exclude'
}

labeled_files = sorted(list(LABELED_DIR.glob("*.h5ad")))
print(f"Found {len(labeled_files)} labeled files.\n")

updated = 0

for file_path in tqdm(labeled_files, desc="Updating tissue_group"):
    adata = sc.read_h5ad(file_path)

    if 'tissue_group' not in adata.obs.columns:
        print(f"  Skipping {file_path.name}: No tissue_group column.")
        continue

    # Apply refined mapping
    adata.obs['tissue_group'] = adata.obs['tissue_group'].map(refined_mapping)

    # Remove spots marked as "Exclude"
    adata = adata[adata.obs['tissue_group'] != 'Exclude'].copy()

    # Save updated file
    adata.write_h5ad(file_path)
    updated += 1

print(f"\n✅ Updated {updated} files with refined grouping.")
print("New groups:", adata.obs['tissue_group'].unique().tolist())

## 3.0 Step 3: Hybrid Pseudo-Labeling Strategy

### Components of the Hybrid Strategy

| Component | Method | Purpose | Priority |
|-----------|--------|---------|----------|
| **1. Marker Gene Rules + Spatial Smoothing** | Expression thresholds from annotated samples + refinement using the spatial KNN graph (`k=6`) | Biologically interpretable and grounded in known marker genes. Spatial smoothing improves consistency within tissue regions. | Primary |
| **2. Reference-based Label Transfer** | Treat the annotated samples as a reference and transfer labels to unannotated samples using joint embedding methods (e.g., `scANVI` or correlation-based transfer) | More robust and data-driven, especially useful when marker genes for a class are weak or non-specific. | Secondary (for robustness) |
| **3. Clustering-based Validation** | Leiden clustering on all samples followed by marker gene overlap mapping | Acts as a fallback to discover patterns not well represented in the limited annotated samples. | Fallback |

### Rationale

This hybrid approach balances:
- **Biological interpretability** (from marker gene rules),
- **Robustness** (from reference-based label transfer), and
- **Spatial coherence** (from the KNN graph).


##  A. Marker Gene Rules + Spatial Smoothing

### Objective

Implement the first component of our hybrid pseudo-labeling strategy. We will assign initial pseudo-labels to unannotated samples using strong marker genes identified from the annotated samples, combined with spatial neighborhood information from the KNN graph.

### Approach

1. **Select robust marker genes** for each refined tissue group (`Tumor`, `Tumor Stroma`, and `Non-Tumor`).
2. **Define scoring rules** based on the expression of these marker genes.
3. **Apply initial labeling** using expression thresholds.
4. **Refine labels** using spatial smoothing (a spot surrounded by Tumor spots is more likely to be labeled as Tumor).
5. **Assign confidence levels** to each pseudo-label (High / Medium / Low) for later filtering.

This component forms the foundation of our pseudo-labels. The subsequent components (Reference-based Label Transfer) will be used to cross-validate and improve these initial labels.

### Define Curated Marker Genes


In [ ]:
# ============================================================
# Curated Marker Genes for Pseudo-Labeling
# ============================================================

marker_genes = {
    'Tumor': [
        'LAPTM4B', 'STMN1', 'NDUFB9', 'COX6C', 'NME2',
        'FASN', 'KRT18', 'MUC1', 'GATA3'
    ],
    'Tumor Stroma': [
        'COL1A1', 'COL1A2', 'TAGLN', 'DCN', 'SFRP2',
        'APOD', 'MYL9', 'CCDC80'
    ],
    'Non-Tumor': [
        # We will use a conservative approach for Non-Tumor
        # (low expression of Tumor + Tumor Stroma markers)
    ]
}

print("Marker genes defined for each group.")

Marker genes defined for each group.


### Scoring Function (Initial Labeling) and Pseudo Label Assignment

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

def compute_marker_scores(adata, marker_genes):
    """
    Compute normalized scores for Tumor, Tumor Stroma, and Non-Tumor
    based on average expression of curated marker genes.
    """
    scores = pd.DataFrame(index=adata.obs_names)

    # Tumor Score
    tumor_genes = [g for g in marker_genes['Tumor'] if g in adata.var_names]
    if tumor_genes:
        tumor_expr = adata[:, tumor_genes].X.toarray().mean(axis=1)
        scores['Tumor_score'] = MinMaxScaler().fit_transform(tumor_expr.reshape(-1, 1)).flatten()
    else:
        scores['Tumor_score'] = 0.0

    # Tumor Stroma Score
    stroma_genes = [g for g in marker_genes['Tumor Stroma'] if g in adata.var_names]
    if stroma_genes:
        stroma_expr = adata[:, stroma_genes].X.toarray().mean(axis=1)
        scores['TumorStroma_score'] = MinMaxScaler().fit_transform(stroma_expr.reshape(-1, 1)).flatten()
    else:
        scores['TumorStroma_score'] = 0.0

    # Non-Tumor Score (inverse of Tumor + Stroma)
    scores['NonTumor_score'] = 1 - ((scores['Tumor_score'] + scores['TumorStroma_score']) / 2)

    return scores


def assign_pseudo_labels(scores, quantile=0.65, margin=0.03):
    """
    Assign pseudo-labels using per-sample quantiles + relative dominance.
    Non-Tumor rule is made more lenient (Option B).
    """
    labels = []
    confidence = []

    # Per-sample quantile thresholds
    tumor_q = scores['Tumor_score'].quantile(quantile)
    stroma_q = scores['TumorStroma_score'].quantile(quantile)

    for _, row in scores.iterrows():
        t = row['Tumor_score']
        s = row['TumorStroma_score']
        nt = row['NonTumor_score']

        # === Tumor ===
        if t >= tumor_q and (t - s) >= margin:
            labels.append('Tumor')
            conf = 'High' if t > 0.80 else 'Medium'

        # === Tumor Stroma ===
        elif s >= stroma_q and (s - t) >= margin:
            labels.append('Tumor Stroma')
            conf = 'High' if s > 0.75 else 'Medium'

        # === Non-Tumor (More Lenient - Option B) ===
        elif nt >= 0.58 and (t + s) < 0.95:
            labels.append('Non-Tumor')
            conf = 'Medium'

        else:
            labels.append('Uncertain')
            conf = 'Low'

        confidence.append(conf)

    return pd.Series(labels, index=scores.index), pd.Series(confidence, index=scores.index)

### Test on one Unnanotated Sample

In [ ]:
import scanpy as sc
from pathlib import Path

# ====================== UPDATE THIS ======================
# Pick one unannotated sample (preferably from Train folder)
sample_name = "GSE203612_GSM6177599"   # ← Change to any unannotated sample
# ========================================================

PROCESSED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train")

adata = sc.read_h5ad(PROCESSED_DIR / f"{sample_name}.h5ad")
print(f"Loaded sample: {sample_name}")
print(f"Number of spots: {adata.n_obs}")
print(f"Number of genes: {adata.n_vars}")

#### Compute Marker scores + Assign Pseudo-label genes


In [ ]:
# Compute scores
scores = compute_marker_scores(adata, marker_genes)

# Assign initial pseudo-labels
pseudo_labels, confidence = assign_pseudo_labels(scores, quantile=0.75, margin=0.08)

# Add to AnnData
adata.obs['pseudo_label'] = pseudo_labels
adata.obs['pseudo_label_confidence'] = confidence

print("\nPseudo-label distribution:")
print(adata.obs['pseudo_label'].value_counts())

print("\nConfidence distribution:")
print(adata.obs['pseudo_label_confidence'].value_counts())

In [ ]:
# Compute marker scores using the updated function
scores = compute_marker_scores(adata, marker_genes)

# Assign pseudo-labels with the NEW relaxed parameters
pseudo_labels, confidence = assign_pseudo_labels(
    scores,
    quantile=0.65,     # ← Relaxed
    margin=0.03        # ← Relaxed
)

# Add to AnnData
adata.obs['pseudo_label'] = pseudo_labels
adata.obs['pseudo_label_confidence'] = confidence

print("=== Updated Relaxed Parameters (quantile=0.65, margin=0.03) ===")
print("\nPseudo-label distribution:")
print(adata.obs['pseudo_label'].value_counts())

print("\nConfidence distribution:")
print(adata.obs['pseudo_label_confidence'].value_counts())

### Process all Unnanotated Samples

In [ ]:
# ============================================================
# Apply Pseudo-Labeling to All Unannotated Samples
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ====================== PATHS ======================
PROCESSED_DIR   = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train")
LABELED_DIR     = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_labeled")
OUTPUT_DIR      = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_pseudo_labeled")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ====================================================

# Get list of already labeled samples (to skip them)
labeled_samples = {f.stem for f in LABELED_DIR.glob("*.h5ad")}

# Get all samples in the processed train folder
all_samples = sorted([f for f in PROCESSED_DIR.glob("*.h5ad")])

print(f"Total samples in train folder: {len(all_samples)}")
print(f"Already labeled samples: {len(labeled_samples)}")
print(f"Samples to pseudo-label: {len(all_samples) - len(labeled_samples)}\n")

successfully_labeled = []
failed = []

for sample_path in tqdm(all_samples, desc="Pseudo-labeling samples"):
    sample_name = sample_path.stem

    # Skip already labeled samples
    if sample_name in labeled_samples:
        continue

    try:
        # Load sample
        adata = sc.read_h5ad(sample_path)

        # Compute marker scores
        scores = compute_marker_scores(adata, marker_genes)

        # Assign pseudo-labels
        pseudo_labels, confidence = assign_pseudo_labels(
            scores,
            quantile=0.65,
            margin=0.03
        )

        # Add to AnnData
        adata.obs['pseudo_label'] = pseudo_labels
        adata.obs['pseudo_label_confidence'] = confidence

        # Save
        output_path = OUTPUT_DIR / f"{sample_name}.h5ad"
        adata.write_h5ad(output_path)

        successfully_labeled.append(sample_name)

    except Exception as e:
        print(f"\nError on {sample_name}: {e}")
        failed.append(sample_name)

# ====================== SUMMARY ======================
print("\n" + "="*70)
print("PSEUDO-LABELING SUMMARY")
print("="*70)
print(f"Successfully pseudo-labeled : {len(successfully_labeled)} samples")
print(f"Failed                        : {len(failed)} samples")

if failed:
    print("\nFailed samples:")
    print(failed)

### Merge Annotated + Pseudo-Labeled Samples

In [ ]:
# ============================================================
# Merge Annotated + Pseudo-Labeled Samples into One Folder
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import shutil
import warnings
warnings.filterwarnings("ignore")

# ====================== PATHS ======================
ANNOTATED_DIR   = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_labeled")
PSEUDO_DIR      = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_pseudo_labeled")
FINAL_DIR       = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled")

FINAL_DIR.mkdir(parents=True, exist_ok=True)
# ====================================================

print("Merging annotated and pseudo-labeled samples...\n")

# --- Process Annotated Samples (24 samples) ---
annotated_files = sorted(list(ANNOTATED_DIR.glob("*.h5ad")))
print(f"Found {len(annotated_files)} annotated samples.")

for file_path in tqdm(annotated_files, desc="Processing annotated samples"):
    adata = sc.read_h5ad(file_path)

    # Standardize columns
    adata.obs['final_label'] = adata.obs['tissue_group']
    adata.obs['label_source'] = 'annotated'
    adata.obs['confidence'] = 'High'   # Annotated labels are considered high confidence

    # Save
    output_path = FINAL_DIR / file_path.name
    adata.write_h5ad(output_path)

# --- Process Pseudo-Labeled Samples ---
pseudo_files = sorted(list(PSEUDO_DIR.glob("*.h5ad")))
print(f"\nFound {len(pseudo_files)} pseudo-labeled samples.")

for file_path in tqdm(pseudo_files, desc="Processing pseudo-labeled samples"):
    adata = sc.read_h5ad(file_path)

    # Standardize columns
    adata.obs['final_label'] = adata.obs['pseudo_label']
    adata.obs['label_source'] = 'pseudo'
    adata.obs['confidence'] = adata.obs['pseudo_label_confidence']

    # Save
    output_path = FINAL_DIR / file_path.name
    adata.write_h5ad(output_path)

print("\n" + "="*70)
print("MERGING COMPLETE")
print("="*70)
print(f"Total samples in final folder: {len(list(FINAL_DIR.glob('*.h5ad')))}")
print(f"Saved to: {FINAL_DIR}")

## Quality Control

### Global Summary of Pseudo-Labels + Annotated Samples



In [ ]:
# ============================================================
# Quality Control: Merged Annotated + Pseudo-Labeled Samples
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

FINAL_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled")

final_files = sorted(list(FINAL_DIR.glob("*.h5ad")))
print(f"Found {len(final_files)} samples in final labeled folder.\n")

all_labels = []
all_confidence = []
all_sources = []
sample_stats = []

for file_path in tqdm(final_files, desc="Analyzing final labeled samples"):
    adata = sc.read_h5ad(file_path)

    label_counts = adata.obs['final_label'].value_counts().to_dict()
    conf_counts = adata.obs['confidence'].value_counts().to_dict()
    source = adata.obs['label_source'].iloc[0]

    sample_stats.append({
        'sample_name': file_path.stem,
        'label_source': source,
        'total_spots': adata.n_obs,
        'Tumor': label_counts.get('Tumor', 0),
        'Tumor Stroma': label_counts.get('Tumor Stroma', 0),
        'Non-Tumor': label_counts.get('Non-Tumor', 0),
        'Uncertain': label_counts.get('Uncertain', 0),
        'High_conf': conf_counts.get('High', 0),
        'Medium_conf': conf_counts.get('Medium', 0),
        'Low_conf': conf_counts.get('Low', 0),
    })

    all_labels.append(adata.obs['final_label'])
    all_confidence.append(adata.obs['confidence'])
    all_sources.append(adata.obs['label_source'])

# Combine everything
label_series = pd.concat(all_labels)
conf_series = pd.concat(all_confidence)
source_series = pd.concat(all_sources)
sample_stats_df = pd.DataFrame(sample_stats)

print("\n" + "="*75)
print("GLOBAL FINAL LABEL DISTRIBUTION (Annotated + Pseudo-labeled)")
print("="*75)
print(label_series.value_counts())

print("\n" + "="*75)
print("CONFIDENCE DISTRIBUTION")
print("="*75)
print(conf_series.value_counts())

print("\n" + "="*75)
print("LABEL SOURCE DISTRIBUTION")
print("="*75)
print(source_series.value_counts())

print("\n" + "="*75)
print("SUMMARY STATISTICS")
print("="*75)
print(f"Total samples              : {len(final_files)}")
print(f"Total spots                : {len(label_series):,}")
print(f"Annotated samples          : {(source_series == 'annotated').sum()}")
print(f"Pseudo-labeled samples     : {(source_series == 'pseudo').sum()}")
print(f"Overall Uncertain rate     : {(label_series == 'Uncertain').mean()*100:.2f}%")

### Identify Sample with High Uncertain Rate

In [ ]:
sample_stats_df['Uncertain_rate'] = sample_stats_df['Uncertain'] / sample_stats_df['total_spots']

print("\nSamples with highest Uncertain rate:")
print(sample_stats_df.sort_values('Uncertain_rate', ascending=False)[['sample_name', 'Uncertain_rate']].head(10))

### Visualize Pseudo-Labels on Few Samples

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pick 2–3 samples to visualize
samples_to_plot = sample_stats_df['sample_name'].sample(3, random_state=42).tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, sample_name in enumerate(samples_to_plot):
    adata = sc.read_h5ad(FINAL_DIR / f"{sample_name}.h5ad")
    coords = adata.obsm['spatial']

    sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=adata.obs['pseudo_label'],
        palette={'Tumor': '#E63946', 'Tumor Stroma': '#F4A261',
                 'Non-Tumor': '#2A9D8F', 'Uncertain': '#ADB5BD'},
        s=8, ax=axes[i], alpha=0.8
    )
    axes[i].set_title(f"{sample_name}\nPseudo-Labels", fontsize=11)
    axes[i].axis('off')
    axes[i].invert_yaxis()

plt.tight_layout()

# Save figure with higher quality
save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "sample_pseudo_label_visualization.png"
plt.savefig(save_path, dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Figure saved to: {save_path}")

### Visualize Label Distribution by Source

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

temp_df = pd.DataFrame({
    'final_label': label_series,
    'label_source': source_series
})

plt.figure(figsize=(10, 5))
sns.countplot(data=temp_df, x='label_source', hue='final_label', palette='Set2')
plt.title("Final Label Distribution by Source (Annotated vs Pseudo)")
plt.xticks(rotation=0)
plt.tight_layout()

# Save figure with higher quality
save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "final_label_distribution_by_source.png"
plt.savefig(save_path, dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Figure saved to: {save_path}")

##  B. Reference-based Label Transfer

### Objective

While the marker gene + spatial smoothing approach has produced reasonable pseudo-labels, it still leaves ~26.6% of spots as "Uncertain", particularly affecting the Non-Tumor class. To improve label coverage and quality, we now incorporate **reference-based label transfer** as the second component of our hybrid strategy.

### What is Reference-based Label Transfer?

We treat the annotated samples as a **high-quality reference**. Using joint embedding and label transfer techniques, we project the unannotated samples into the same latent space as the reference and transfer labels probabilistically. This approach can capture more complex patterns than simple marker gene thresholding and is especially useful when marker genes for certain classes (e.g., Non-Tumor) are weak or non-specific.

### Method Chosen

We will use **`scANVI`** (from the `scvi-tools` package), which is currently one of the strongest methods for semi-supervised label transfer in single-cell/spatial transcriptomics. It learns a joint latent representation while explicitly using the available labels from the reference.

### Why Add This Component?

- Improves labeling of spots that were previously marked as "Uncertain".
- Provides better coverage for the Non-Tumor class.
- Acts as a second opinion that can be combined with the rule-based labels.
- Increases the total number of usable training samples for the downstream multimodal model.

### Implementation Plan

1. Prepare a combined AnnData containing both annotated (reference) and unannotated samples.
2. Train an `scANVI` model using the annotated samples as labeled data.
3. Transfer labels to the unannotated samples.
4. Compare and potentially combine the `scANVI` predictions with the earlier rule-based pseudo-labels.
5. Perform quality control on the final labels.

In [ ]:
# Install scvi-tools (this may take a few minutes)
!pip install scvi-tools --quiet

import scvi
print("scvi-tools version:", scvi.__version__)

### Data Preparation

In [ ]:
# ============================================================
# Corrected Data Preparation for scANVI (Refinement Mode)
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import scvi
print("scvi-tools version:", scvi.__version__)

# ====================== PATHS ======================
FINAL_LABELED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled")
# ====================================================

print("Loading all samples...\n")

adata_list = []
sample_names = []

for file_path in tqdm(sorted(FINAL_LABELED_DIR.glob("*.h5ad")), desc="Loading samples"):
    adata = sc.read_h5ad(file_path)

    # Add batch column
    adata.obs['batch'] = file_path.stem

    # === Create labels column for scANVI ===
    # Rule:
    # - If final_label is NOT "Uncertain" → keep the label (annotated or pseudo)
    # - If final_label == "Uncertain"     → set to "Unknown" (scANVI will predict these)

    adata.obs['labels'] = adata.obs['final_label'].astype(str)

    # Mark only Uncertain spots as Unknown
    adata.obs.loc[adata.obs['final_label'] == 'Uncertain', 'labels'] = 'Unknown'

    adata_list.append(adata)
    sample_names.append(file_path.stem)

# Concatenate all samples
print("\nConcatenating all samples...")
adata_combined = sc.concat(
    adata_list,
    join='outer',
    label='sample',
    keys=sample_names
)

print(f"\nCombined AnnData shape: {adata_combined.shape}")

# Clean up labels column
adata_combined.obs['labels'] = adata_combined.obs['labels'].replace('nan', 'Unknown')

print("\nLabel distribution for scANVI:")
print(adata_combined.obs['labels'].value_counts())

# Convert to proper Categorical (important for scvi)
adata_combined.obs['labels'] = pd.Categorical(
    adata_combined.obs['labels'],
    categories=['Non-Tumor', 'Tumor', 'Tumor Stroma', 'Unknown']
)

print("\nFinal label categories:", adata_combined.obs['labels'].cat.categories.tolist())

# Basic gene filtering
sc.pp.filter_genes(adata_combined, min_cells=10)
print(f"\nAfter filtering: {adata_combined.shape[1]} genes remaining")

# Save combined object
output_path = FINAL_LABELED_DIR.parent / "train_combined_for_scanvi_refined.h5ad"
adata_combined.write_h5ad(output_path)
print(f"\n✅ Combined AnnData saved to: {output_path}")

### Clean Labels + Setup scANVI

In [ ]:
# ============================================================
# Clean NaNs + Train scVI + scANVI
# ============================================================

import scanpy as sc
import numpy as np
import scvi
from scvi.model import SCVI, SCANVI
import warnings
warnings.filterwarnings("ignore")

print("Loading data...")
adata = sc.read_h5ad("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_combined_for_scanvi_refined.h5ad")

print(f"Original shape: {adata.shape}")

# ====================== AGGRESSIVE NAN CLEANING ======================
print("\nChecking and cleaning NaN/Inf values in .X...")

# Convert to dense if sparse (for safety)
if hasattr(adata.X, "toarray"):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()

# Replace NaN and Inf
n_nans = np.isnan(X_dense).sum()
n_infs = np.isinf(X_dense).sum()

print(f"Found {n_nans} NaNs and {n_infs} Infs")

X_dense = np.nan_to_num(X_dense, nan=0.0, posinf=0.0, neginf=0.0)

# Put cleaned matrix back
adata.X = X_dense

print("NaN/Inf values removed. Matrix is now clean.")

# ====================== SETUP scVI ======================
print("\nSetting up scVI...")
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="batch",
    labels_key="labels"
)

# More stable configuration
scvi_model = SCVI(
    adata,
    n_layers=2,
    n_latent=15,           # Further reduced for stability
    gene_likelihood="nb"
)

print("Training scVI model...")
scvi_model.train(
    max_epochs=60,
    early_stopping=True,
    early_stopping_patience=8
)
print("✅ scVI training complete.")

# ====================== scANVI ======================
print("\nTraining scANVI...")
scanvi_model = SCANVI.from_scvi_model(
    scvi_model,
    labels_key="labels",
    unlabeled_category="Unknown"
)

scanvi_model.train(
    max_epochs=35,
    early_stopping=True,
    early_stopping_patience=6
)
print("\n✅ scANVI training complete!")

### Predict Labels with scANVI

In [ ]:
import scanpy as sc
import numpy as np
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

print("Loading data...")

adata = sc.read_h5ad("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_combined_for_scanvi_refined.h5ad")

print(f"Original shape: {adata.shape}")

# ====================== AGGRESSIVE NAN CLEANING (Re-applied) ======================
print("\nChecking and cleaning NaN/Inf values in .X (before scANVI prediction)...")

# Convert to dense if sparse (for safety)
if hasattr(adata.X, "toarray"):
    X_dense = adata.X.toarray()
else:
    X_dense = adata.X.copy()

# Replace NaN and Inf
n_nans = np.isnan(X_dense).sum()
n_infs = np.isinf(X_dense).sum()

print(f"Found {n_nans} NaNs and {n_infs} Infs")

X_dense = np.nan_to_num(X_dense, nan=0.0, posinf=0.0, neginf=0.0)

# Put cleaned matrix back
adata.X = X_dense

print("NaN/Inf values removed. Matrix is now clean for prediction.")

# The scANVI model should already be set up from the training phase.
# Explicitly calling setup_anndata again here is likely redundant and can cause issues.
# scvi-tools will attempt to transfer the setup from the model if needed.

print("\nGetting soft predictions (more stable method)...")

# Get soft probabilities instead of hard predictions
soft_preds = scanvi_model.predict(adata, soft=True)

# Convert to hard labels
predictions = soft_preds.idxmax(axis=1).values
max_probs = soft_preds.max(axis=1).values

# Add to AnnData
adata.obs['scanvi_predicted_label'] = predictions
adata.obs['scanvi_max_probability'] = max_probs

print("\nPrediction distribution:")
print(adata.obs['scanvi_predicted_label'].value_counts())

print("\nConfidence summary:")
print(adata.obs['scanvi_max_probability'].describe())

# ====================== SAVE BACK ======================
print("\nSaving predictions to individual files...")

FINAL_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled")
UPDATED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled_updated")
UPDATED_DIR.mkdir(parents=True, exist_ok=True)

for sample_name in tqdm(adata.obs['batch'].unique(), desc="Updating files"):
    mask = adata.obs['batch'] == sample_name
    preds = adata.obs.loc[mask, ['scanvi_predicted_label', 'scanvi_max_probability']]

    original_path = FINAL_DIR / f"{sample_name}.h5ad"
    if not original_path.exists():
        continue

    sample_adata = sc.read_h5ad(original_path)

    sample_adata.obs['scanvi_predicted_label'] = preds['scanvi_predicted_label'].values
    sample_adata.obs['scanvi_max_probability'] = preds['scanvi_max_probability'].values

    # Create final label (replace Uncertain with scANVI prediction)
    sample_adata.obs['final_combined_label'] = sample_adata.obs['final_label'].astype(str)
    uncertain_mask = sample_adata.obs['final_label'] == 'Uncertain'
    sample_adata.obs.loc[uncertain_mask, 'final_combined_label'] = preds.loc[uncertain_mask, 'scanvi_predicted_label']

    sample_adata.write_h5ad(UPDATED_DIR / f"{sample_name}.h5ad")

print("\n✅ Done! Updated files saved to:", UPDATED_DIR)

### Comparison Analysis

In [ ]:
# ============================================================
# Comparison: final_label vs scanvi_predicted_label
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

UPDATED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled_updated")

updated_files = sorted(list(UPDATED_DIR.glob("*.h5ad")))
print(f"Found {len(updated_files)} updated sample files.\n")

# Containers for global analysis
comparison_records = []
uncertain_reassignment = []

for file_path in tqdm(updated_files, desc="Comparing labels"):
    adata = sc.read_h5ad(file_path)

    # Only analyze samples that have scANVI predictions
    if 'scanvi_predicted_label' not in adata.obs.columns:
        continue

    df = adata.obs[['final_label', 'scanvi_predicted_label', 'scanvi_max_probability']].copy()

    # Ensure both columns are string type before comparison to avoid Categorical TypeError
    df['final_label'] = df['final_label'].astype(str)
    df['scanvi_predicted_label'] = df['scanvi_predicted_label'].astype(str)

    # Overall agreement
    agreement = (df['final_label'] == df['scanvi_predicted_label']).mean()

    comparison_records.append({
        'sample_name': file_path.stem,
        'total_spots': len(df),
        'agreement_rate': agreement,
        'num_uncertain_original': (df['final_label'] == 'Uncertain').sum()
    })

    # Focus on previously Uncertain spots
    uncertain_mask = df['final_label'] == 'Uncertain'
    if uncertain_mask.sum() > 0:
        reassigned = df.loc[uncertain_mask, 'scanvi_predicted_label'].value_counts()
        for label, count in reassigned.items():
            uncertain_reassignment.append({
                'sample_name': file_path.stem,
                'reassigned_to': label,
                'count': count,
                'avg_confidence': df.loc[uncertain_mask & (df['scanvi_predicted_label'] == label), 'scanvi_max_probability'].mean()
            })

# ====================== RESULTS ======================
print("\n" + "="*70)
print("OVERALL COMPARISON RESULTS")
print("="*70)

comparison_df = pd.DataFrame(comparison_records)
print(f"Average agreement rate across samples : {comparison_df['agreement_rate'].mean()*100:.2f}%")
print(f"Total previously Uncertain spots      : {comparison_df['num_uncertain_original'].sum():,}")

# Reassignment of Uncertain spots
reassignment_df = pd.DataFrame(uncertain_reassignment)
if not reassignment_df.empty:
    print("\n" + "="*70)
    print("HOW scANVI RE-ASSIGNED PREVIOUSLY 'UNCERTAIN' SPOTS")
    print("="*70)

    reassignment_summary = reassignment_df.groupby('reassigned_to').agg({
        'count': 'sum',
        'avg_confidence': 'mean'
    }).sort_values('count', ascending=False)

    reassignment_summary['percentage'] = (reassignment_summary['count'] / reassignment_summary['count'].sum() * 100).round(2)
    print(reassignment_summary)

### Create Final Combined Label

In [ ]:
# ============================================================
# Create Final Combined Label
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

UPDATED_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_labeled_updated")
FINAL_OUTPUT_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_combined")
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

updated_files = sorted(list(UPDATED_DIR.glob("*.h5ad")))
print(f"Found {len(updated_files)} updated sample files.\n")

for file_path in tqdm(updated_files, desc="Creating final combined labels"):
    adata = sc.read_h5ad(file_path)

    # Create new columns, ensuring confidence starts as string to avoid categorical errors
    adata.obs['final_combined_label'] = adata.obs['final_label'].astype(str)
    adata.obs['final_label_confidence'] = 'High'

    # Apply decision rule
    uncertain_mask = adata.obs['final_label'] == 'Uncertain'
    high_conf_mask = adata.obs['scanvi_max_probability'] >= 0.80

    # Case 1: Previously Uncertain + scANVI is confident → Use scANVI prediction
    replace_mask = uncertain_mask & high_conf_mask
    adata.obs.loc[replace_mask, 'final_combined_label'] = adata.obs.loc[replace_mask, 'scanvi_predicted_label']
    adata.obs.loc[replace_mask, 'final_label_confidence'] = 'High'

    # Case 2: Previously Uncertain + scANVI is NOT confident → Keep as Uncertain
    keep_uncertain_mask = uncertain_mask & ~high_conf_mask
    adata.obs.loc[keep_uncertain_mask, 'final_combined_label'] = 'Uncertain'
    # Ensure final_label_confidence can accept 'Low' if it's not already a category
    adata.obs.loc[keep_uncertain_mask, 'final_label_confidence'] = adata.obs.loc[keep_uncertain_mask, 'final_label_confidence'].astype(str)
    adata.obs.loc[keep_uncertain_mask, 'final_label_confidence'] = 'Low'

    # Save
    output_path = FINAL_OUTPUT_DIR / file_path.name
    adata.write_h5ad(output_path)

print(f"\n✅ Final combined labels created and saved to: {FINAL_OUTPUT_DIR}")

# ====================== Quick Summary ======================
print("\n" + "="*70)
print("FINAL COMBINED LABEL DISTRIBUTION (All Samples)")
print("="*70)

# Quick global summary
all_labels = []
for file_path in FINAL_OUTPUT_DIR.glob("*.h5ad"):
    adata = sc.read_h5ad(file_path)
    all_labels.append(adata.obs['final_combined_label'])

combined = pd.concat(all_labels)
print(combined.value_counts())
print(f"\nTotal spots: {len(combined):,}")

In [ ]:
# ============================================================
# Clean "nan" values (Write to New Folder - Recommended)
# ============================================================

import scanpy as sc
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# ====================== PATHS ======================
INPUT_DIR  = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_combined")
OUTPUT_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_combined_cleaned")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ====================================================

updated_files = sorted(list(INPUT_DIR.glob("*.h5ad")))
print(f"Found {len(updated_files)} files to clean.\n")

for file_path in tqdm(updated_files, desc="Cleaning files"):
    adata = sc.read_h5ad(file_path)

    # Convert to string to avoid Categorical issues
    adata.obs['final_combined_label'] = adata.obs['final_combined_label'].astype(str)

    # Replace "nan" and NaN with "Uncertain"
    adata.obs.loc[adata.obs['final_combined_label'].isin(['nan', 'NaN']), 'final_combined_label'] = 'Uncertain'

    # Clean confidence column
    if 'final_label_confidence' in adata.obs.columns:
        adata.obs['final_label_confidence'] = adata.obs['final_label_confidence'].astype(str)
        adata.obs.loc[adata.obs['final_combined_label'] == 'Uncertain', 'final_label_confidence'] = 'Low'

    # Save to NEW folder (this avoids file locking issues)
    output_path = OUTPUT_DIR / file_path.name
    adata.write_h5ad(output_path)

print("\n✅ Cleaning complete.")
print(f"Cleaned files saved to: {OUTPUT_DIR}")

# ====================== Verification ======================
all_labels = []
for file_path in OUTPUT_DIR.glob("*.h5ad"):
    adata = sc.read_h5ad(file_path)
    all_labels.append(adata.obs['final_combined_label'])

combined = pd.concat(all_labels)
print("\nFinal distribution after cleaning:")
print(combined.value_counts())

### Quality Control Visualizations

In [ ]:
# Global Distribution of Final Labels

import scanpy as sc
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

FINAL_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_combined_cleaned")

all_labels = []
all_confidence = []

for file_path in tqdm(sorted(FINAL_DIR.glob("*.h5ad")), desc="Loading labels"):
    adata = sc.read_h5ad(file_path)
    all_labels.append(adata.obs['final_combined_label'])
    if 'final_label_confidence' in adata.obs.columns:
        all_confidence.append(adata.obs['final_label_confidence'])

label_series = pd.concat(all_labels)
conf_series = pd.concat(all_confidence) if all_confidence else None

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Label Distribution
sns.countplot(x=label_series, ax=axes[0], palette='Set2', order=label_series.value_counts().index)
axes[0].set_title("Final Combined Label Distribution", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Label")
axes[0].tick_params(axis='x', rotation=30)

# Confidence Distribution (if available)
if conf_series is not None:
    sns.countplot(x=conf_series, ax=axes[1], palette='Set1')
    axes[1].set_title("Final Label Confidence Distribution", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Confidence Level")

plt.tight_layout()

# Save figure
save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "final_combined_label_and_confidence_distribution.png"
plt.savefig(save_path, dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()

print("\nLabel Distribution:")
print(label_series.value_counts(normalize=True).round(4) * 100)
print(f"\n✅ Figure saved to: {save_path}")

In [ ]:
# Spatial Visualization of Final Labels

import scanpy as sc
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

FINAL_DIR = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/data/processed/st_preprocessed/train_final_combined_cleaned")

# Select a few representative samples to visualize
samples_to_plot = [
    "GSE213688_GSM6592051",                            # Annotated sample
    "Human_Breast_He_06222020_ST_BC23272_E2",          # Pseudo-labeled sample
    "Human_Breast_Wu_06052021_Visium_CID4465"          # Another sample
]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, sample_name in enumerate(samples_to_plot):
    file_path = FINAL_DIR / f"{sample_name}.h5ad"

    if not file_path.exists():
        print(f"File not found: {sample_name}")
        continue

    adata = sc.read_h5ad(file_path)
    coords = adata.obsm['spatial']

    # Color palette
    palette = {
        'Tumor': '#E63946',
        'Tumor Stroma': '#F4A261',
        'Non-Tumor': '#2A9D8F',
        'Uncertain': '#ADB5BD'
    }

    sns.scatterplot(
        x=coords[:, 0],
        y=coords[:, 1],
        hue=adata.obs['final_combined_label'],
        palette=palette,
        s=12,
        alpha=0.85,
        ax=axes[i]
    )

    axes[i].set_title(f"{sample_name}\nFinal Combined Labels", fontsize=11, fontweight='bold')
    axes[i].axis('off')
    axes[i].invert_yaxis()
    axes[i].legend(title="Label", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()

# Save figure
save_dir = Path("/content/drive/My Drive/MSC Project/SpaHisto-Net/visualizations/pseudo_labeling")
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / "spatial_final_labels_selected_samples.png"
plt.savefig(save_path, dpi=1200, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✅ Figure saved to: {save_path}")